In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q gradio

In [3]:
import json
import os
import ast

def extract_classes_from_notebook(path, class_names):
    ######
    with open(path, 'r', encoding='utf-8') as f:
        nb = json.load(f)
    full_code = ""
    for cell in nb['cells']:
        if cell['cell_type'] == 'code':
            full_code += "".join(cell['source']) + "\n"
    tree = ast.parse(full_code)
    class_defs = [node for node in tree.body if isinstance(node, ast.ClassDef) and (node.name in class_names or 'Block' in node.name)]
    new_tree = ast.Module(body=class_defs, type_ignores=[])
    namespace = {}
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    namespace.update({'torch': torch, 'nn': nn, 'F': F})
    exec(compile(new_tree, filename="<ast>", mode="exec"), namespace)
    return {name: namespace.get(name) for name in class_names}
    #### this funciton was written by gemini

notebook_dir = '/content/drive/MyDrive/cs320/final project/'

try:
    lenet_data = extract_classes_from_notebook(os.path.join(notebook_dir, 'LeNet-5.ipynb'), ['LeNet5', 'RBFLayer'])
    resnet_data = extract_classes_from_notebook(os.path.join(notebook_dir, 'Resnet.ipynb'), ['ResidualCNN', 'ResidualBlock'])
    modern_data = extract_classes_from_notebook(os.path.join(notebook_dir, 'Modern CNN.ipynb'), ['ModernCNN'])

    RBFLayer = lenet_data['RBFLayer']
    LeNet5RBF = lenet_data['LeNet5']
    ResidualCNN = resnet_data['ResidualCNN']
    ModernCNN = modern_data['ModernCNN']

except Exception as e:
    print(f"Extraction failed: {e}")

In [5]:
import gradio as gr
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms

MODEL_DIR = '/content/drive/MyDrive/cs320/final project/models/'
RESNET_PATH = f'{MODEL_DIR}resnet_byclass_final.pth'
MODERN_CNN_PATH = f'{MODEL_DIR}modern_byclass_final.pth'
LENET_PATH = f'{MODEL_DIR}lenet5_rbf_byclass_final.pth'

inference_transform = transforms.Compose([ ## written by gemini
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

def load_pytorch_model(model_path, model_class, *args, **kwargs): # autocompleted by Gemini from here to ...
    try:
        model = model_class(*args, **kwargs) ## here
        model.load_state_dict(torch.load(model_path, map_location=torch.device('cpu')))
        model.eval()
        return model
    except Exception as e:
        return None

lenet_model = load_pytorch_model(LENET_PATH, LeNet5RBF, num_classes=62)
resnet_model = load_pytorch_model(RESNET_PATH, ResidualCNN, num_classes=62)
modern_cnn_model = load_pytorch_model(MODERN_CNN_PATH, ModernCNN, num_classes=62)

CLASS_LABELS = [str(i) for i in range(10)] + [chr(i) for i in range(65, 91)] + [chr(i) for i in range(97, 123)] ## Written by Gemini

def preprocess_image(image):
    if image is None: return None, None
    if isinstance(image, dict):
        image = image.get('composite')
    if image is None: return None, None ## written by gemini

    img = Image.fromarray(image).convert('L')
    img_inverted = Image.fromarray(255 - np.array(img))

    tensor_standard = inference_transform(img_inverted).unsqueeze(0)
    img_transposed = img_inverted.transpose(Image.TRANSPOSE) ## written by gemini
    tensor_transposed = inference_transform(img_transposed).unsqueeze(0) ## written by gemini

    return tensor_standard, tensor_transposed

def predict_handwriting(image):
    t_std, t_trans = preprocess_image(image)
    if t_std is None: return "", "", "" ## written by gemini

    with torch.no_grad():
        res_p = CLASS_LABELS[torch.argmax(resnet_model(t_trans), 1).item()] if resnet_model else "N/A"
        mod_p = CLASS_LABELS[torch.argmax(modern_cnn_model(t_trans), 1).item()] if modern_cnn_model else "N/A"
        len_p = CLASS_LABELS[torch.argmax(lenet_model(t_std), 1).item()] if lenet_model else "N/A"
    return res_p, mod_p, len_p

with gr.Blocks() as demo:
    gr.Markdown("# Interactive Handwriting Recognition")
    with gr.Row():
        canvas = gr.Sketchpad(label="Draw here", image_mode="L")
        with gr.Column():
            resnet_output = gr.Textbox(label="ResNet (Transposed)")
            modern_cnn_output = gr.Textbox(label="Modern CNN (Transposed)")
            lenet_output = gr.Textbox(label="LeNet-5 (Standard)")
    canvas.change(fn=predict_handwriting, inputs=canvas, outputs=[resnet_output, modern_cnn_output, lenet_output]) ## written by gemini

demo.launch(debug=True) ## written by gemini

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b25050ffdf5cabe83f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


content_type multipart/form-data; boundary=----WebKitFormBoundaryF49bWBWdCqmNWDeI
content_type multipart/form-data; boundary=----WebKitFormBoundaryb1IFnq8Ysblmv9DH
content_type multipart/form-data; boundary=----WebKitFormBoundaryuWpCjaRmv2SlPQEf
content_type multipart/form-data; boundary=----WebKitFormBoundaryQf5xusXbsTDvFRdv
content_type multipart/form-data; boundary=----WebKitFormBoundaryTB39e9XKmZkk9cNf
content_type multipart/form-data; boundary=----WebKitFormBoundary6Fg46b9lATAnIotn
content_type multipart/form-data; boundary=----WebKitFormBoundaryqARM2McY73y6V4Jr
content_type multipart/form-data; boundary=----WebKitFormBoundarykluRrgV2ATvQbNBq
content_type multipart/form-data; boundary=----WebKitFormBoundaryuawXVyLzAmRXnZiU
content_type multipart/form-data; boundary=----WebKitFormBoundaryqPUOTtYTyfv4fWfw
content_type multipart/form-data; boundary=----WebKitFormBoundaryBfGvuk54B3wzwmZK
content_type multipart/form-data; boundary=----WebKitFormBoundaryBZSKujB0jqKl5f61
content_type mul